# 03 — Paper Figures

Produce publication-ready figures and Table 1 for the arXiv preprint.
Figures are saved to `paper/figures/` at 300 dpi.

## Setup

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Use a publication-ready style.
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 10,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'legend.fontsize': 8,
    'figure.dpi': 150,
})
sns.set_palette('colorblind')

RESULTS_DIR = Path('../results/full/mid')
FIG_DIR = Path('../paper/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

summaries = [json.loads(p.read_text()) for p in sorted(RESULTS_DIR.glob('*_summary.json'))]
df = pd.DataFrame(summaries)
agg = df.groupby(['strategy', 'dataset']).agg(
    mean_sr=('success_rate', 'mean'),
    std_sr=('success_rate', 'std'),
    mean_cost=('total_cost_usd', 'mean'),
).reset_index()

print(f'{len(df)} runs loaded')

## Figure 1 — Headline Pareto Frontier

Aggregate success rate vs. mean cost across all four datasets.

In [ ]:
headline = df.groupby('strategy').agg(
    mean_sr=('success_rate', 'mean'),
    mean_cost=('total_cost_usd', 'mean'),
).reset_index()

fig, ax = plt.subplots(figsize=(5, 4))
colors = sns.color_palette('colorblind', n_colors=len(headline))
for (_, row), color in zip(headline.iterrows(), colors):
    ax.scatter(row['mean_cost'], row['mean_sr'], color=color,
               label=row['strategy'], s=80, zorder=3)
    ax.annotate(row['strategy'], (row['mean_cost'], row['mean_sr']),
                textcoords='offset points', xytext=(5, 3), fontsize=7)

ax.set_xlabel('Mean cost per task (USD)')
ax.set_ylabel('Mean success rate')
ax.set_title('Headline Pareto — All Datasets')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('$%.3f'))
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig1_headline_pareto.pdf', bbox_inches='tight')
plt.show()

## Figure 2 — Heterogeneity Effect

Pairwise comparison of `vanilla_debate` vs. `hetero_debate` per dataset.

In [ ]:
hetero = agg[agg['strategy'].isin(['vanilla_debate', 'hetero_debate'])].copy()
fig, ax = plt.subplots(figsize=(6, 3.5))
sns.barplot(data=hetero, x='dataset', y='mean_sr',
            hue='strategy', ax=ax, capsize=0.05)
ax.set_xlabel('Dataset')
ax.set_ylabel('Mean success rate')
ax.set_title('Figure 2 — Heterogeneity Effect')
ax.legend(title='Strategy')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig2_heterogeneity.pdf', bbox_inches='tight')
plt.show()

## Figure 3 — Per-Dataset Breakdown

In [ ]:
datasets = agg['dataset'].unique()
fig, axes = plt.subplots(1, len(datasets), figsize=(13, 3.5), sharey=False)
for ax, dataset in zip(axes, datasets):
    sub = agg[agg['dataset'] == dataset]
    sns.barplot(data=sub, x='strategy', y='mean_sr', ax=ax, palette='colorblind')
    ax.set_title(dataset)
    ax.set_xlabel('')
    ax.set_ylabel('Success rate' if ax is axes[0] else '')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Figure 3 — Per-Dataset Strategy Breakdown', fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3_per_dataset.pdf', bbox_inches='tight')
plt.show()

## Table 1 — Strategy × Dataset × Budget

LaTeX-ready table of mean success rate ± std.

In [ ]:
table_df = agg.copy()
table_df['SR'] = table_df.apply(
    lambda r: f"{r['mean_sr']:.3f} ± {r['std_sr']:.3f}" if pd.notna(r['std_sr']) else f"{r['mean_sr']:.3f}",
    axis=1
)
pivot_table = table_df.pivot(index='strategy', columns='dataset', values='SR')
print(pivot_table.to_latex(escape=False))
pivot_table